# Inspect KS4 Amplitude-Driven Merge Candidates

This notebook checks whether a KS4 cluster looks like a single waveform shape with multiple amplitude modes.

It does three things:
- ranks candidate clusters with strong amplitude bimodality,
- shows the detailed diagnostic for one cluster,
- compares that cluster's template with its most similar partner.


In [1]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.postprocess.ks4_diagnostics import (
    diagnose_ks4_cluster,
    rank_ks4_amplitude_split_candidates,
)


In [2]:
RESULTS_DIR = Path("sorting_temp/sake_day24/Kilosort4_2026-03-07_003728")
TOP_K = 15
MIN_SPIKES = 250
MIN_BALANCE = 0.10

# Set to an integer to inspect a specific cluster.
# Leave as None to inspect the top-ranked candidate automatically.
CLUSTER_ID = None

RESULTS_DIR.resolve()

PosixPath('/local/workdir/ys2375/PreprocessPipeline/sorting_temp/sake_day24/Kilosort4_2026-03-07_003728')

In [3]:
def load_ks4_arrays(results_dir: Path) -> dict[str, np.ndarray]:
    arrays = {}
    for name in [
        "spike_clusters.npy",
        "amplitudes.npy",
        "templates.npy",
        "similar_templates.npy",
        "spike_times.npy",
    ]:
        path = results_dir / name
        if path.exists():
            arrays[name] = np.load(path)
    return arrays


def cluster_amplitudes(arrays: dict[str, np.ndarray], cluster_id: int) -> np.ndarray:
    spike_clusters = arrays["spike_clusters.npy"].astype(int)
    amplitudes = arrays["amplitudes.npy"].astype(float)
    return amplitudes[spike_clusters == int(cluster_id)]


def cluster_spike_times_sec(arrays: dict[str, np.ndarray], cluster_id: int, sample_rate: float) -> np.ndarray:
    spike_clusters = arrays["spike_clusters.npy"].astype(int)
    spike_times = arrays["spike_times.npy"].astype(float)
    return spike_times[spike_clusters == int(cluster_id)] / float(sample_rate)


def template_peak_channel(template: np.ndarray) -> int:
    return int(np.argmax(np.max(np.abs(template), axis=0)))


def normalized_template_corr(template_a: np.ndarray, template_b: np.ndarray) -> float:
    a = np.asarray(template_a, dtype=float).reshape(-1)
    b = np.asarray(template_b, dtype=float).reshape(-1)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return float("nan")
    return float(np.dot(a, b) / (na * nb))


def plot_cluster_amplitude_hist(amplitudes: np.ndarray, diag) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(amplitudes, bins=80, color="0.35", alpha=0.85)
    ax.set_title(f"Cluster {diag.cluster_id}: amplitude histogram")
    ax.set_xlabel("Amplitude")
    ax.set_ylabel("Spike count")
    if diag.amplitude_centers is not None:
        ax.axvline(diag.amplitude_centers[0], color="tab:blue", linestyle="--", label="low center")
        ax.axvline(diag.amplitude_centers[1], color="tab:orange", linestyle="--", label="high center")
    if diag.split_amplitude_threshold is not None:
        ax.axvline(diag.split_amplitude_threshold, color="tab:red", linestyle=":", label="split threshold")
    ax.legend()
    plt.show()


def plot_cluster_amplitude_vs_time(times_sec: np.ndarray, amplitudes: np.ndarray, diag) -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(times_sec, amplitudes, s=2, alpha=0.25, color="tab:green")
    ax.set_title(f"Cluster {diag.cluster_id}: amplitude across time")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    if diag.split_amplitude_threshold is not None:
        ax.axhline(diag.split_amplitude_threshold, color="tab:red", linestyle=":", label="split threshold")
        ax.legend()
    plt.show()


def plot_template_pair(templates: np.ndarray, cluster_id: int, other_cluster_id: int | None) -> None:
    if other_cluster_id is None:
        print("No matching cluster available.")
        return

    template_a = templates[int(cluster_id)]
    template_b = templates[int(other_cluster_id)]
    ch_a = template_peak_channel(template_a)
    ch_b = template_peak_channel(template_b)
    corr = normalized_template_corr(template_a, template_b)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    axes[0].plot(template_a[:, ch_a], color="tab:blue")
    axes[0].set_title(f"cluster {cluster_id} peak ch {ch_a}")
    axes[0].set_xlabel("Sample")
    axes[0].set_ylabel("Template value")
    axes[1].plot(template_b[:, ch_b], color="tab:orange")
    axes[1].set_title(f"cluster {other_cluster_id} peak ch {ch_b}")
    axes[1].set_xlabel("Sample")
    fig.suptitle(f"Normalized template corr = {corr:.3f}")
    plt.tight_layout()
    plt.show()


In [ ]:
candidate_diags = rank_ks4_amplitude_split_candidates(
    RESULTS_DIR,
    min_spikes=MIN_SPIKES,
    min_balance=MIN_BALANCE,
    top_k=TOP_K,
)

candidate_df = pd.DataFrame([diag.__dict__ for diag in candidate_diags])
candidate_df

In [ ]:
arrays = load_ks4_arrays(RESULTS_DIR)
sample_rate = 20000.0

if CLUSTER_ID is None:
    if not candidate_diags:
        raise RuntimeError("No candidate clusters found.")
    CLUSTER_ID = int(candidate_diags[0].cluster_id)

diag = diagnose_ks4_cluster(RESULTS_DIR, CLUSTER_ID)
pd.Series(diag.__dict__)

In [ ]:
amps = cluster_amplitudes(arrays, diag.cluster_id)
times_sec = cluster_spike_times_sec(arrays, diag.cluster_id, sample_rate=sample_rate)

plot_cluster_amplitude_hist(amps, diag)
plot_cluster_amplitude_vs_time(times_sec, amps, diag)


In [ ]:
templates = arrays["templates.npy"]
plot_template_pair(templates, diag.cluster_id, diag.best_matching_cluster_id)

## How to read the output

- `amplitude_centers`: two amplitude modes estimated from a 1D two-cluster split.
- `amplitude_balance`: fraction of spikes in the smaller mode. If this is tiny, the split is weak.
- `amplitude_separation`: distance between the two amplitude modes in log-amplitude units, normalized by pooled variance.
- `amplitude_explained_fraction`: how much better the two-mode fit is than a single-mode fit.
- `best_template_correlation`: similarity to the most similar other cluster after template normalization.

A suspicious amplitude-driven merge usually looks like this:
- high `amplitude_explained_fraction`,
- decent `amplitude_balance`,
- high `best_template_correlation`,
- nearly identical peak-channel waveforms, but split mostly by amplitude.
